In [1]:
!pip install -q pyts

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 24.9 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (Conv2D, MaxPooling2D, Flatten, Dense, 
                                      Dropout, Input, BatchNormalization, 
                                      GlobalAveragePooling2D)
from tensorflow.keras.applications import ResNet50  # Issue 1: Pre-trained teacher
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import (classification_report, confusion_matrix, 
                              accuracy_score, precision_score, recall_score, f1_score)
from pyts.image import GramianAngularField

2026-02-27 13:58:55.536372: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772200735.820009      17 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772200735.896508      17 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772200736.546330      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772200736.546381      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772200736.546385      17 computation_placer.cc:177] computation placer alr

In [3]:
# Bỏ qua hàng đầu tiên, dùng hàng 2 làm header
df = pd.read_csv("/kaggle/input/datasets/hmn6969/datasetswat/SWaT.csv",header=1, low_memory=False)

In [4]:
time_candidates = [c for c in df.columns if 'time' in c.lower() or 'timestamp' in c.lower()]
if not time_candidates:
    timestamp_col = df.columns[0]
else:
    timestamp_col = time_candidates[0]

df = df[~df[timestamp_col].astype(str).str.lower().eq(timestamp_col.lower())]
df[timestamp_col] = pd.to_datetime(df[timestamp_col], errors="coerce", utc=True)
df = df.dropna(subset=[timestamp_col])
df = df.set_index(timestamp_col)

print("Cột thời gian được sử dụng:", timestamp_col)

Cột thời gian được sử dụng: GMT +0


/tmp/ipykernel_17/1033324384.py:8: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[timestamp_col] = pd.to_datetime(df[timestamp_col], errors="coerce", utc=True)


In [5]:
attack_periods = [
    # Attack 1: 15h (GMT+8) -> 7h (GMT+0)
    ('2019-07-20 07:08:46', '2019-07-20 07:10:31'),
    # Attack 2: 15h (GMT+8) -> 7h (GMT+0)
    ('2019-07-20 07:15:00', '2019-07-20 07:19:32'),
    # Attack 3: 15h (GMT+8) -> 7h (GMT+0)
    ('2019-07-20 07:26:57', '2019-07-20 07:30:48'),
    # Attack 4: 15h (GMT+8) -> 7h (GMT+0)
    ('2019-07-20 07:38:50', '2019-07-20 07:46:20'),
    # Attack 5: 15h (GMT+8) -> 7h (GMT+0)
    ('2019-07-20 07:54:00', '2019-07-20 07:56:00'),
    # Attack 6: 16h (GMT+8) -> 8h (GMT+0)
    ('2019-07-20 08:02:56', '2019-07-20 08:16:18')
]

attack_datetime_periods = [
    (pd.to_datetime(start).tz_localize("UTC"), pd.to_datetime(end).tz_localize("UTC"))
    for start, end in attack_periods
]

df['Attack'] = 0
for start, end in attack_datetime_periods:
    df.loc[start:end, 'Attack'] = 1

print("Số mẫu Attack:", df['Attack'].sum())
print("Tỷ lệ Attack:", df['Attack'].mean() * 100)

Số mẫu Attack: 1981
Tỷ lệ Attack: 13.210189383835688


In [6]:
plt.rcParams['figure.figsize'] = (15, 5)
status_cols = [col for col in df.columns if df[col].astype(str).str.contains("Active|Inactive", case=False).any()]

print("\n Các cột có Active/Inactive:", status_cols)

for col in status_cols:
    df[col] = df[col].map({'Active': 1, 'Inactive': 0})

# print(df[status_cols].head())

target_column = 'Attack'
feature_columns = df.columns.drop(target_column)

for col in feature_columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

feature_std = df[feature_columns].std()
useless_columns = feature_std[feature_std == 0].index

if not useless_columns.empty:
    print("\nCác cột không có biến thiên (std=0) và có thể loại bỏ:")
    print(list(useless_columns))
    print("Số cột bị loại bỏ:", len(useless_columns))
    # df.drop(columns=useless_columns, inplace=True)
else:
    print("\nKhông có cột nào bị loại bỏ do không có biến thiên.")


 Các cột có Active/Inactive: ['LS 201', 'LS 202', 'LSL 203', 'LSLL 203', 'LS 401', 'LSH 601', 'LSH 602', 'LSH 603', 'LSL 601', 'LSL 602', 'LSL 603']

Các cột không có biến thiên (std=0) và có thể loại bỏ:
['LS 201', 'LS 202', 'LSL 203', 'LSLL 203', 'AIT 401', 'LS 401', 'LSH 603', 'LSL 601', 'LSL 602']
Số cột bị loại bỏ: 9


In [7]:
#---------------------------------------------------------------------------------
# Select top features based on correlation with 'Attack'

selected_features = ['FIT 101', 'LIT 101', 'MV 101', 'P1_STATE', 'P101 Status', 'AIT 201', 'AIT 202', 'AIT 203', 'FIT 201', 'MV201', 'P203 Status', 'P205 Status',
                    'AIT 301', 'AIT 302', 'AIT 303', 'DPIT 301', 'FIT 301', 'LIT 301', 'MV 301', 'MV 302', 'MV 303', 'MV 304', 'P3_STATE', 'P301 Status', 
                    'AIT 402', 'FIT 401', 'LIT 401', 'P401 Status', 'UV401', 'AIT 501', 'AIT 502', 'AIT 503', 'AIT 504', 'FIT 501', 'FIT 502', 'FIT 503',
                    'FIT 504', 'MV 501', 'PIT 501', 'PIT 502', 'PIT 503', 'FIT 601', 'LSH 601', 'P601 Status']


df_model = df[selected_features + ['Attack']].copy()

print(f"Đã tạo DataFrame mới với {len(selected_features)} đặc trưng được chọn.")

print(df_model.head())

Đã tạo DataFrame mới với 44 đặc trưng được chọn.
                                  FIT 101   LIT 101  MV 101  P1_STATE  \
GMT +0                                                                  
2019-07-20 04:30:00+00:00             0.0  729.8658       1         3   
2019-07-20 04:30:01+00:00             0.0  729.4340       1         3   
2019-07-20 04:30:02.004013+00:00      0.0  729.1200       1         3   
2019-07-20 04:30:03.004013+00:00      0.0  728.6882       1         3   
2019-07-20 04:30:04+00:00             0.0  727.7069       1         3   

                                  P101 Status     AIT 201   AIT 202  \
GMT +0                                                                
2019-07-20 04:30:00+00:00                   2  142.527557  9.293002   
2019-07-20 04:30:01+00:00                   2  142.527557  9.293002   
2019-07-20 04:30:02.004013+00:00            2  142.527557  9.293002   
2019-07-20 04:30:03.004013+00:00            2  142.527557  9.289157   
2019-07-20 04

In [8]:
#---------------------------------------------------------------------------------
# missing value imputation
print(f"Số giá trị NaN trước khi xử lý: {df_model.isnull().sum().sum()}")

df_model.ffill(inplace=True)

df_model.bfill(inplace=True)

print(f"Số giá trị NaN sau khi xử lý: {df_model.isnull().sum().sum()}")

Số giá trị NaN trước khi xử lý: 0
Số giá trị NaN sau khi xử lý: 0


In [9]:
#- ---------------------------------------------------------------------------
# Split data into train and test sets based on time

from sklearn.model_selection import train_test_split

split_timestamp = pd.to_datetime('2019-07-20 07:00:00').tz_localize("UTC")

df_train = df_model.loc[df_model.index <= split_timestamp]

df_test = df_model.loc[df_model.index > split_timestamp]

print("split timestamp:", split_timestamp)
print(f"Train set: {df_train.shape}, Test set: {df_test.shape}")


split timestamp: 2019-07-20 07:00:00+00:00
Train set: (8996, 45), Test set: (6000, 45)


In [10]:
# label distribution

print("\n--- Phân bố nhãn trong tập train ---")
print(df_train['Attack'].value_counts())

print("\n--- Phân bố nhãn trong tập test ---")
print(df_test['Attack'].value_counts())

X_train = df_train[selected_features]
y_train = df_train['Attack']

X_test = df_test[selected_features]
y_test = df_test['Attack']

print("Completed data preprocessing and splitting!")



--- Phân bố nhãn trong tập train ---
Attack
0    8996
Name: count, dtype: int64

--- Phân bố nhãn trong tập test ---
Attack
0    4019
1    1981
Name: count, dtype: int64
Completed data preprocessing and splitting!


In [11]:
# =============================================================================
# Phase 1: Non-overlapping Segments (thay thế Sliding Window)
# =============================================================================

def create_non_overlapping_segments(X, cycle_length=100):
    """
    Cắt dữ liệu thành các đoạn không chồng lấp có độ dài cycle_length.
    Loại bỏ phần dư ở cuối chuỗi nếu không đủ tạo thành 1 chu kỳ.
    
    Input:  X shape (N, F)  — N timesteps, F features
    Output: segments shape (num_segments, cycle_length, F)
    """
    N = X.shape[0]
    num_segments = N // cycle_length
    usable = num_segments * cycle_length
    X_trimmed = X[:usable]
    segments = X_trimmed.reshape(num_segments, cycle_length, -1)
    return segments

def align_labels(y, cycle_length=100):
    """
    Cắt nhãn cho khớp với số đoạn, gán nhãn 1 cho đoạn nếu có BẤT KỲ
    điểm Attack nào trong đoạn đó.
    Trả về:
      - y_segment: nhãn mỗi đoạn (num_segments,)
      - y_pointwise: nhãn gốc đã trim (num_segments * cycle_length,)
    """
    N = len(y)
    num_segments = N // cycle_length
    usable = num_segments * cycle_length
    y_trimmed = y[:usable]
    y_reshaped = y_trimmed.reshape(num_segments, cycle_length)
    y_segment = (y_reshaped.max(axis=1) > 0).astype(int)
    return y_segment, y_trimmed

print("Phase 1: Hàm tạo Non-overlapping Segments đã sẵn sàng.")

Phase 1: Hàm tạo Non-overlapping Segments đã sẵn sàng.


In [12]:
 # =============================================================================
# Phase 2: GAF Encoding — Per-Sensor Univariate GAF (Issue 5 fix)
# =============================================================================

def encode_single_sensor_gaf(segments_1d, method='summation'):
    """
    Biến đổi Gramian Angular Field cho MỘT sensor (univariate).
    
    Input:  segments_1d shape (num_segments, cycle_length)
    Output: gaf_images  shape (num_segments, cycle_length, cycle_length, 1)
    """
    cycle_length = segments_1d.shape[1]
    gasf = GramianAngularField(image_size=cycle_length, method=method)
    gaf = gasf.fit_transform(segments_1d)          # (num_segments, cycle_length, cycle_length)
    return gaf[:, :, :, np.newaxis]                 # thêm chiều channel


def gaf_to_3ch(gaf_1ch):
    """
    Nhân đôi kênh đơn thành 3 kênh để tương thích với ResNet50 pre-trained.
    Input:  (N, H, W, 1)
    Output: (N, H, W, 3)
    """
    return np.repeat(gaf_1ch, 3, axis=-1)

print("Phase 2: Hàm mã hóa GAF per-sensor (univariate) đã sẵn sàng.")

Phase 2: Hàm mã hóa GAF per-sensor (univariate) đã sẵn sàng.


In [13]:
# =============================================================================
# Phase 3: Knowledge Distillation — Pre-trained Teacher & Student (Issue 1 fix)
# =============================================================================

def build_teacher_model(input_shape):
    """
    Teacher Model: ResNet50 pre-trained trên ImageNet, đóng băng trọng số.
    Input shape: (cycle_length, cycle_length, 3)   ← 3 kênh (nhân đôi từ GAF 1-kênh)
    """
    base_model = ResNet50(
        weights='imagenet',          # Load trọng số ImageNet — FIX Issue 1
        include_top=False,
        input_shape=input_shape
    )
    # Đóng băng SAU KHI load trọng số đã pre-train
    base_model.trainable = False
    
    teacher = Model(inputs=base_model.input, outputs=base_model.output, name='Teacher')
    return teacher


def build_student_model(input_shape, teacher_output_shape):
    """
    Student Model: Mạng Conv2D nhỏ hơn, đầu ra khớp với Teacher output shape.
    Được huấn luyện để bắt chước Teacher trên dữ liệu Normal.
    """
    inputs = Input(shape=input_shape)
    
    x = Conv2D(16, (3, 3), activation='relu', padding='same')(inputs)
    x = BatchNormalization()(x)
    x = Conv2D(16, (3, 3), activation='relu', padding='same')(x)
    x = MaxPooling2D((2, 2))(x)
    
    x = Conv2D(32, (3, 3), activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = Conv2D(32, (3, 3), activation='relu', padding='same')(x)
    x = MaxPooling2D((2, 2))(x)
    
    x = Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    
    # Adaptive pooling / resize để khớp spatial dims của Teacher output
    # ResNet50 output spatial dims phụ thuộc vào input size
    target_h, target_w, target_c = teacher_output_shape
    x = tf.keras.layers.Resizing(target_h, target_w)(x)
    x = Conv2D(target_c, (1, 1), activation='linear', padding='same')(x)
    
    model = Model(inputs, x, name='Student')
    model.compile(optimizer='adam', loss='mse')
    
    return model

print("Phase 3: Pre-trained Teacher (ResNet50) & Student models đã sẵn sàng.")

Phase 3: Pre-trained Teacher (ResNet50) & Student models đã sẵn sàng.


In [14]:
# =============================================================================
# Phase 4: Training Pipeline — Per-Sensor KD (Issues 2, 3, 5 fixes)
# =============================================================================

def train_kd_single_sensor(train_1d, test_1d, cycle_length=100,
                           epochs=50, batch_size=32, teacher=None):
    """
    Huấn luyện KD cho MỘT sensor (univariate).
    
    Params
    ------
    train_1d : np.ndarray, shape (N_train,)  ← CHỈ dữ liệu Normal (Issue 2)
    test_1d  : np.ndarray, shape (N_test,)
    teacher  : pre-trained Teacher model (shared, sẽ build 1 lần bên ngoài)

    Returns
    -------
    student, gaf_test_3ch, scaler
    """
    # 1. Scale sang [-1, 1] — FIX Issue 3
    scaler = MinMaxScaler(feature_range=(-1, 1))
    train_scaled = scaler.fit_transform(train_1d.reshape(-1, 1)).ravel()
    test_scaled  = scaler.transform(test_1d.reshape(-1, 1)).ravel()
    
    # 2. Non-overlapping segments
    train_segs = create_non_overlapping_segments(
        train_scaled.reshape(-1, 1), cycle_length).squeeze(-1)    # (n_seg, cycle_length)
    test_segs  = create_non_overlapping_segments(
        test_scaled.reshape(-1, 1), cycle_length).squeeze(-1)
    
    # 3. GAF (univariate) + replicate to 3 channels
    gaf_train = gaf_to_3ch(encode_single_sensor_gaf(train_segs))   # (n, H, W, 3)
    gaf_test  = gaf_to_3ch(encode_single_sensor_gaf(test_segs))
    
    # 4. Teacher features (target cho Student)
    teacher_features = teacher.predict(gaf_train, verbose=0)
    teacher_out_shape = teacher_features.shape[1:]  # (h, w, c)
    
    # 5. Build & train Student
    input_shape = (cycle_length, cycle_length, 3)
    student = build_student_model(input_shape, teacher_out_shape)
    
    early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    student.fit(
        gaf_train, teacher_features,
        epochs=epochs, batch_size=batch_size,
        validation_split=0.1, callbacks=[early_stop],
        verbose=0
    )
    
    return student, gaf_test, scaler

print("Phase 4: Per-sensor training pipeline (normal-only, scale [-1,1]) đã sẵn sàng.")

Phase 4: Per-sensor training pipeline (normal-only, scale [-1,1]) đã sẵn sàng.


In [15]:
# =============================================================================
# Phase 5: Anomaly Detection — Cosine Similarity Maps + Algorithm 2 (Issues 4, 6, 7)
# =============================================================================

# ---- Issue 4 FIX: Cosine similarity thay vì MSE ----
def compute_anomaly_maps(teacher, student, gaf_images):
    """
    Tính anomaly map bằng cosine similarity (1 - cos_sim) theo Equation (1) của paper.
    Higher = more anomalous.
    """
    teacher_out = teacher.predict(gaf_images, verbose=0)
    student_out = student.predict(gaf_images, verbose=0)
    
    eps = 1e-8
    dot_product = np.sum(teacher_out * student_out, axis=-1)      # (N, H, W)
    norm_t      = np.linalg.norm(teacher_out, axis=-1) + eps      # (N, H, W)
    norm_s      = np.linalg.norm(student_out, axis=-1) + eps      # (N, H, W)
    cosine_sim  = dot_product / (norm_t * norm_s)
    anomaly_maps = 1.0 - cosine_sim   # trong [0, 2]; càng cao càng bất thường
    return anomaly_maps


# ---- Issue 6 FIX: Thêm segment-length check vào Algorithm 2 ----
def algorithm2_time_series_mapping(anomaly_maps, cycle_length, T, H,
                                   mean_train_len=None, std_train_len=None):
    """
    Algorithm 2 từ paper — có bổ sung kiểm tra độ dài segment (Issue 6).
    
    Nếu mean_train_len và std_train_len được cung cấp:
      - Tính L = mean_train_len + 2 * std_train_len
      - Nếu cycle_length >= L → toàn bộ segment được đánh dấu anomaly
    """
    num_segments = anomaly_maps.shape[0]
    map_h, map_w = anomaly_maps.shape[1], anomaly_maps.shape[2]
    
    y_pred_all = np.zeros(num_segments * cycle_length, dtype=int)
    
    # Issue 6: Kiểm tra segment-length anomaly
    if mean_train_len is not None and std_train_len is not None:
        L = mean_train_len + 2 * std_train_len
    else:
        L = None
    
    for seg_idx in range(num_segments):
        # Issue 6: Nếu cycle_length vượt ngưỡng → đánh dấu toàn segment là anomaly
        if L is not None and cycle_length >= L:
            start_g = seg_idx * cycle_length
            end_g   = start_g + cycle_length
            y_pred_all[start_g:end_g] = 1
            continue
        
        amap = anomaly_maps[seg_idx]
        
        # Resize anomaly map về cycle_length x cycle_length nếu cần
        if map_h != cycle_length or map_w != cycle_length:
            from scipy.ndimage import zoom
            zoom_h = cycle_length / map_h
            zoom_w = cycle_length / map_w
            amap = zoom(amap, (zoom_h, zoom_w), order=1)
        
        map_size = amap.shape[0]
        
        for i in range(map_size):
            if amap[i, i] > T:
                S = np.sum(amap[i, :]) + np.sum(amap[:, i]) - amap[i, i]
                threshold = (2 * map_size - 1) * H * T
                if S > threshold:
                    global_idx = seg_idx * cycle_length + i
                    if global_idx < len(y_pred_all):
                        y_pred_all[global_idx] = 1
    
    return y_pred_all


def get_attack_intervals(y_true):
    """Trả về danh sách các khoảng thời gian tấn công."""
    diff = np.diff(y_true, prepend=0)
    starts = np.where(diff == 1)[0]
    ends = np.where(diff == -1)[0] - 1
    if len(starts) > len(ends):
        ends = np.append(ends, len(y_true) - 1)
    return list(zip(starts, ends))


# Issue 7 FIX: Loại bỏ extend_attack_labels — đánh giá trên nhãn gốc
# (Hàm được giữ lại nhưng KHÔNG sử dụng trong pipeline chính)
def extend_attack_labels(y_true, extension_seconds=600):
    """[DEPRECATED — không dùng cho đánh giá chính, chỉ để phân tích latency]"""
    y_extended = y_true.copy()
    intervals = get_attack_intervals(y_true)
    for start, end in intervals:
        extended_end = min(end + extension_seconds, len(y_true) - 1)
        y_extended[end + 1:extended_end + 1] = 1
    return y_extended


def compute_attack_metrics(y_true, y_pred):
    """Tính Precision, Recall, F1 point-wise."""
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    f1   = 2 * prec * rec / (prec + rec + 1e-9)
    return prec, rec, f1

print("Phase 5: Cosine Similarity Anomaly Maps & Algorithm 2 (đầy đủ) đã sẵn sàng.")

Phase 5: Cosine Similarity Anomaly Maps & Algorithm 2 (đầy đủ) đã sẵn sàng.


In [16]:
# =============================================================================
# Phase 6: Main Pipeline — Per-Sensor KD (all sensors, no stages) + Grid Search
# =============================================================================

CYCLE_LENGTH = 100

# --- 1. Chuẩn bị nhãn test (point-wise) — KHÔNG mở rộng (Issue 7 fix) ---
y_test_vals = y_test.values
num_test_segments = len(y_test_vals) // CYCLE_LENGTH
usable_test = num_test_segments * CYCLE_LENGTH
y_test_trimmed = y_test_vals[:usable_test]

print(f"Cycle length: {CYCLE_LENGTH}")
print(f"Test segments: {num_test_segments}, Usable points: {usable_test}")
print(f"Attack points (raw ground-truth): {y_test_trimmed.sum()}")

# --- 2. Lọc dữ liệu train CHỈ Normal (Issue 2 fix) ---
X_train_normal = X_train[y_train == 0]
print(f"\nTraining trên {len(X_train_normal)} mẫu Normal "
      f"(đã loại {(y_train == 1).sum()} mẫu Attack)")

# --- 3. Build Teacher 1 lần (shared, frozen) ---
input_shape_3ch = (CYCLE_LENGTH, CYCLE_LENGTH, 3)
teacher = build_teacher_model(input_shape_3ch)
teacher.summary()

dummy = np.zeros((1, *input_shape_3ch), dtype=np.float32)
teacher_out_shape = teacher.predict(dummy, verbose=0).shape[1:]
print(f"Teacher output shape: {teacher_out_shape}")

# --- 4. Train per-sensor trên TOÀN BỘ selected_features (không chia stage) ---
T_values = [0.01, 0.02, 0.05, 0.08, 0.1, 0.15, 0.2]
H_values = [0.3, 0.5, 0.7, 0.9, 1.0, 1.2, 1.5]

sensor_report = []
final_pred = np.zeros(usable_test, dtype=int)

print(f"\nTraining {len(selected_features)} sensors...")
print(f"{'='*60}")

for idx, sensor in enumerate(selected_features):
    print(f"\n[{idx+1}/{len(selected_features)}] Sensor: {sensor}")
    
    train_1d = X_train_normal[sensor].values
    test_1d  = X_test[sensor].values
    
    # Train KD cho sensor
    try:
        student, gaf_test_3ch, scaler = train_kd_single_sensor(
            train_1d, test_1d, cycle_length=CYCLE_LENGTH,
            epochs=50, batch_size=32, teacher=teacher
        )
    except Exception as e:
        print(f"  [SKIP] Lỗi: {e}")
        continue
    
    # Tính Anomaly Maps (cosine similarity)
    anomaly_maps = compute_anomaly_maps(teacher, student, gaf_test_3ch)
    print(f"  Anomaly Maps: shape={anomaly_maps.shape}, "
          f"min={anomaly_maps.min():.6f}, max={anomaly_maps.max():.6f}, "
          f"mean={anomaly_maps.mean():.6f}")
    
    # Grid Search T, H
    best_f1 = -1
    best_pred_sensor = np.zeros(usable_test, dtype=int)
    best_cfg = (0, 0)
    best_metrics = (0, 0, 0)
    
    for T in T_values:
        for H in H_values:
            pred = algorithm2_time_series_mapping(
                anomaly_maps, CYCLE_LENGTH, T, H)
            min_len = min(len(y_test_trimmed), len(pred))
            p, r, f1 = compute_attack_metrics(
                y_test_trimmed[:min_len], pred[:min_len])
            if f1 > best_f1 and r > 0:
                best_f1 = f1
                best_pred_sensor = pred
                best_cfg = (T, H)
                best_metrics = (p, r, f1)
    
    sensor_report.append({
        'Sensor': sensor, 'T': best_cfg[0], 'H': best_cfg[1],
        'Precision': best_metrics[0], 'Recall': best_metrics[1], 'F1': best_metrics[2]
    })
    print(f"  Best T={best_cfg[0]}, H={best_cfg[1]}, "
          f"P={best_metrics[0]:.4f} R={best_metrics[1]:.4f} F1={best_metrics[2]:.4f}")
    
    # OR vào final prediction
    min_len = min(len(final_pred), len(best_pred_sensor))
    final_pred[:min_len] = np.bitwise_or(
        final_pred[:min_len], best_pred_sensor[:min_len])

# --- 5. Kết quả cuối ---
min_len = min(len(y_test_trimmed), len(final_pred))
fin_p, fin_r, fin_f1 = compute_attack_metrics(
    y_test_trimmed[:min_len], final_pred[:min_len])

print("\n" + "=" * 60)
print("SENSOR REPORT (GAF + Knowledge Distillation)")
print("=" * 60)
for r in sensor_report:
    print(f"  {r['Sensor']:<16} | T={r['T']:<6} H={r['H']:<6} | "
          f"P={r['Precision']:.4f} R={r['Recall']:.4f} F1={r['F1']:.4f}")

print("\n" + "=" * 60)
print(f"FINAL ENSEMBLE RESULT (OR of all {len(selected_features)} sensors)")
print(f"  Precision : {fin_p:.4f}")
print(f"  Recall    : {fin_r:.4f}")
print(f"  F1-Score  : {fin_f1:.4f}")
print("=" * 60)

# Confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test_trimmed[:min_len], final_pred[:min_len]))
print("\nClassification Report:")
print(classification_report(y_test_trimmed[:min_len], final_pred[:min_len],
                            target_names=['Normal', 'Attack']))

Cycle length: 100
Test segments: 60, Usable points: 6000
Attack points (raw ground-truth): 1981

Training trên 8996 mẫu Normal (đã loại 0 mẫu Attack)


2026-02-27 13:59:31.065649: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


Model: "Teacher"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 100, 100,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 106, 106,  │          0 │ input_layer[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 50, 50,    │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 50, 50,    │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 50, 50,    │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 52, 52,    │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 25, 25,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 25, 25,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 25, 25,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 25, 25,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 25, 25,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 25, 25,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 25, 25,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 25, 25,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 25, 25,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 25, 25,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 25, 25,    │      1,024 │ conv2_block1_3_c

 Total params: 23,587,712 (89.98 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 23,587,712 (89.98 MB)

Teacher output shape: (4, 4, 2048)

Training 44 sensors...

[1/44] Sensor: FIT 101
  Anomaly Maps: shape=(60, 4, 4), min=0.099456, max=0.649398, mean=0.367035
  Best T=0.01, H=0.3, P=0.3302 R=1.0000 F1=0.4964

[2/44] Sensor: LIT 101
  Anomaly Maps: shape=(60, 4, 4), min=0.057074, max=0.736446, mean=0.416960
  Best T=0.01, H=0.3, P=0.3302 R=1.0000 F1=0.4964

[3/44] Sensor: MV 101
  Anomaly Maps: shape=(60, 4, 4), min=0.083796, max=0.548484, mean=0.359363
  Best T=0.01, H=0.3, P=0.3302 R=1.0000 F1=0.4964

[4/44] Sensor: P1_STATE
  Anomaly Maps: shape=(60, 4, 4), min=0.068795, max=0.600052, mean=0.391894
  Best T=0.01, H=0.3, P=0.3302 R=1.0000 F1=0.4964

[5/44] Sensor: P101 Status
  Anomaly Maps: shape=(60, 4, 4), min=0.092850, max=0.626367, mean=0.384729
  Best T=0.01, H=0.3, P=0.3302 R=1.0000 F1=0.4964

[6/44] Sensor: AIT 201
  Anomaly Maps: shape=(60, 4, 4), min=0.081698, max=0.651459, mean=0.352503
  Best T=0.01, H=0.3, P=0.3302 R=1.0000 F1=0.4964

[7/44] Sensor: AIT 202
  Anomaly Map

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
